## Phase 3: Deep Learning for Text Classification

In this phase, we apply deep learning-based models to the support ticket classification task. While classical machine learning approaches achieved reasonable performance, their improvements through hyperparameter tuning were limited. Deep learning models are therefore explored to better capture semantic relationships within the text data. The main objectives are:

- Load and preprocess the cleaned data
- Build a vocabulary and tokenize text
- Prepare a train/test spllit and a DataLoader
- Pad sequences to fixed length- Compare with ML baseline approach
- Implement an LSTM model- Optimize through hyperparameter tuning
- Train and evaluate the model

#### Load and preprocess the cleaned data
We use the cleaned dataset from Phase 1. The text data is stored in the 'clean_text' column and the target labels (departments) are in the 'queue' column.

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/dataset_en_clean.csv")
texts = df["clean_text"].values

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(df["queue"].values)

#### Tokenization and Vocabulary Construction
To process textual data with neural networks, we need to transform raw text into numerical representations. We tokenize each ticket by splitting it into words. Since text has been cleaned during Phase 1, a simple whitespace-based tokenizer is sufficient. We then construct a vocabulary from all text data. Each word is mapped to a unique integer index, transforming each text into a sequence of integers.

In [2]:
from collections import Counter
import torch

'''
Takes an input string and returns a list of the words in the input string

Args:
    - text: A string, containing a customer support ticket body
Return:
    - A list of words (strings) that were contained in the input string
'''
def tokenize(text: str) -> list[str]:
    return text.split()

counter = Counter()
for text in texts:
    counter.update(tokenize(text)) # Count how often each word appears in all of the ticket bodies

# Define a vocabulary, where the most common words a mapped to indices staring from 2
vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(20000))}
vocab["<PAD>"] = 0 # Padding is mapped to 0
vocab["<UNK>"] = 1 # Unknown is mapped to 1

We now use our dictionary to encode each ticket in our dataset:

In [3]:
'''
Tokenizes an input text, converts each word to its numerical ID and returns a list of said IDs

Args:
    - text: A string, containing a customer support ticket body
    - vocab: The previously established vocabulary (a dictionary)
Return:
    - Returns a list of integer, representing the input string in numerical form
'''
def encode(text: str, vocab: dict) -> list[int]:
    return [vocab.get(word, vocab["<UNK>"]) for word in tokenize(text)]

encoded_texts = [encode(text, vocab) for text in texts]

#### Padding and Truncation
Neural networks require inputs of fixed size, but text sequences vary in length. To enable batch processing, all sequences must be transformed to the same length. From Phase 1 analysis, the median token count is around 40. We select a maximum sequence length of 80 tokens to capture most messages while balancing computational efficiency. Sequences longer than this are truncated; shorter ones are padded with zeros.

In [4]:
from torch.nn.utils.rnn import pad_sequence

max_len = 80

'''
Takes a sequence of encoded tokens and returns it unchanged, padded or truncated to fit a given max lenght

Args:
    - sequence: a list of integers containing encoded tokens of one text 
    - max_len: an integer showing the max lenght for the output sequence
Return:
    - The trucaned or padded input sequence
'''
def pad_truncate(sequence: list[int], max_len: int) -> list[int]:
    if len(sequence) > max_len:
        return sequence[:max_len]
    else:
        return sequence + [0] * (max_len - len(sequence))

padded_sequences = [pad_truncate(seq, max_len) for seq in encoded_texts] # trucante or pad each sequence in the list of encoded texts
padded_sequences = torch.tensor(padded_sequences) # transform list of sequences to tensor

#### Preparing the train and test data
Like in phase 2, we create a train/test split using the scikit-learn test/train split. 

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

We wrap the padded sequences and labels in a custom PyTorch `Dataset` so the training loop can access samples by index. The `DataLoader` then batches and shuffles the data for training and creates a deterministic iterator for evaluation.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TicketDataset(Dataset):
    """
    Dataset wrapper for tokenized ticket sequences and integer labels.
    """
    def __init__(self, X, y):
        """
        Store input sequences and labels as tensors.

        Args:
            - X: Tensor of shape (n_samples, seq_len) with token indices.
            - y: Array-like of integer labels.
        Returns:
            - None. The constructor stores tensors in-place.
        """
        self.X = X
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        """
        Return the number of samples in the dataset.

        Args:
            - None.
        Returns:
            - Integer number of samples in the dataset.
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        Return a single (sequence, label) pair by index.

        Args:
            - idx: Integer index of the sample to retrieve.
        Returns:
            - Tuple (sequence_tensor, label_tensor) for the given index.
        """
        return self.X[idx], self.y[idx]

# Build train/test datasets
train_ds = TicketDataset(X_train, y_train)
test_ds = TicketDataset(X_test, y_test)

# DataLoaders for batching and shuffling
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

#### LSTM-based Text Classification Model

To capture the sequential and contextual structure of the input text, we use a Long Short-Term Memory (LSTM) network.The model consists of three main components:
1. An embedding layer that maps word indices to dense vector representations.
2. An LSTM layer that encodes the sequence of word embeddings into a fixed-size representation.
3. A fully connected layer that maps the sequence representation to class scores.

The final hidden state of the LSTM is used as a compact representation of the input text and serves as input to the classification layer.

In [7]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    """
    LSTM-based text classifier with embedding, LSTM encoder, and linear output layer.
    """
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_classes: int,
        dropout: float = 0.3,
        bidirectional: bool = True
    ) -> None:
        """
        Initialize model layers and configure the embedding, recurrent encoder, and classifier head.

        Args:
            - vocab_size: Size of vocabulary.
            - embedding_dim: Dimension of word embeddings.
            - hidden_dim: Hidden size of LSTM.
            - num_classes: Number of output classes.
            - dropout: Dropout probability before classifier.
            - bidirectional: Whether to use a bidirectional LSTM.
        Returns:
            - None. The constructor initializes the model components in-place.
        """
        super().__init__()

        self.bidirectional = bidirectional

        # Map token indices to dense vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # Sequence encoder
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional
        )

        # Regularize the sequence representation
        self.dropout = nn.Dropout(dropout)

        # Classification head
        fc_in_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_in_dim, num_classes)

    def forward(self, tensor: torch.Tensor) -> torch.Tensor:
        """
        Run a forward pass through embedding, LSTM encoder, and linear classifier.

        Args:
            - tensor: LongTensor of shape (batch_size, seq_len) with token indices.
        Returns:
            - Logits tensor of shape (batch_size, num_classes) suitable for classification loss.
        """
        # Embed tokens
        embedded = self.embedding(tensor)

        # Encode sequence; use final hidden state
        _, (hidden, _) = self.lstm(embedded)

        if self.bidirectional:
            # Concatenate last forward and backward hidden states
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]

        hidden = self.dropout(hidden)
        out = self.fc(hidden)
        return out


#### Configure training and run the loop
We select the device and initialize the model. Then we compute class weights, set up the weighted loss and optimizer, and run the training loop. The loop iterates over epochs, performs forward and backward passes, applies gradient clipping, and reports loss.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Select device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize LSTM model and move to device
model = LSTMClassifier(len(vocab), 128, 128, len(set(labels))).to(device)

# Compute class weights to handle class imbalance
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

# Normalize class weights and move to device
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights = class_weights / class_weights.mean()

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(75):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        # Move batch to device
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass and optimization
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

    # Log epoch loss
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 836.5474
Epoch 2, Loss: 727.5281
Epoch 3, Loss: 641.5601
Epoch 4, Loss: 559.6423
Epoch 5, Loss: 477.2980
Epoch 6, Loss: 422.1362
Epoch 7, Loss: 360.6683
Epoch 8, Loss: 310.6355
Epoch 9, Loss: 266.6982
Epoch 10, Loss: 230.3023
Epoch 11, Loss: 202.9017
Epoch 12, Loss: 174.8341
Epoch 13, Loss: 149.1557
Epoch 14, Loss: 125.2380
Epoch 15, Loss: 106.9801
Epoch 16, Loss: 94.4945
Epoch 17, Loss: 79.9693
Epoch 18, Loss: 68.1729
Epoch 19, Loss: 60.5397
Epoch 20, Loss: 51.6051
Epoch 21, Loss: 47.6955
Epoch 22, Loss: 40.1303
Epoch 23, Loss: 38.3032
Epoch 24, Loss: 34.8464
Epoch 25, Loss: 35.4589
Epoch 26, Loss: 35.8218
Epoch 27, Loss: 27.7561
Epoch 28, Loss: 25.3967
Epoch 29, Loss: 27.1284
Epoch 30, Loss: 25.4139
Epoch 31, Loss: 27.8561
Epoch 32, Loss: 20.1538
Epoch 33, Loss: 22.0159
Epoch 34, Loss: 22.7749
Epoch 35, Loss: 24.3892
Epoch 36, Loss: 22.3393
Epoch 37, Loss: 20.9204
Epoch 38, Loss: 21.3852
Epoch 39, Loss: 17.1306
Epoch 40, Loss: 19.2988
Epoch 41, Loss: 20.2714
Epoch 42, 

#### Evaluate the trained model

Despite extensive tuning and experimentation, the LSTM-based model did not outperform the classical machine learning approach based on TF-IDF. This can be attributed to several factors, including the relatively limited dataset size, the strong class imbalance, and the effectiveness of TF-IDF representations for keyword-driven classification tasks such as support ticket categorization.

In [9]:
from sklearn.metrics import classification_report

# Set model to evaluation mode (disables dropout and batch norm)
model.eval()

# Initialize lists to store predictions and true labels
all_preds = []
all_labels = []

# Evaluate model on test set without computing gradients
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        # Move batch to device (CPU or GPU)
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass to get model outputs (logits)
        outputs = model(X_batch)
        
        # Get predicted class indices by taking argmax over logits
        preds = torch.argmax(outputs, dim=1)
        
        # Append predictions and true labels to lists for later evaluation
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

# Generate classification report with per-class metrics
report = classification_report(
    all_labels,
    all_preds,
    labels=list(range(len(label_encoder.classes_))),
    target_names=label_encoder.classes_,
    zero_division=0
)
print(report)

                                 precision    recall  f1-score   support

           billing and payments       0.79      0.79      0.79       319
               customer service       0.50      0.54      0.52       482
                general inquiry       0.61      0.40      0.49        47
                human resources       0.58      0.54      0.56        70
                     it support       0.57      0.53      0.55       388
                product support       0.50      0.52      0.51       615
          returns and exchanges       0.50      0.40      0.44       164
            sales and pre-sales       0.62      0.44      0.51       103
service outages and maintenance       0.75      0.60      0.67       133
              technical support       0.61      0.67      0.64       947

                       accuracy                           0.59      3268
                      macro avg       0.61      0.54      0.57      3268
                   weighted avg       0.59      0

In [13]:
model_path = "lstm_classifier_state.pt"
torch.save(model.state_dict(), model_path)
print(f"Saved model state to {model_path}")

Saved model state to lstm_classifier_state.pt
